# Train Hybrid Spatial-Frequency (HSF) trên Kaggle GPU

**Chuẩn bị (1 lần):**
1. Settings → Accelerator: **GPU T4 x2 hoặc P100**
2. Add Input → dataset benchmark của bạn (bộ chứa `ForenSynths/ForenSynths/train`, GANGen-Detection, UniversalFakeDetect, DiffusionForensics, Diffusion1kStep)
3. Run All. Kết quả tải về ở mục Output: `hybrid_best.pth`, `results/` (metrics + confusion matrices)

Thời gian dự kiến: ~10-15 phút/epoch trên T4 → ~2 tiếng cho 10 epochs + ~40 phút đánh giá 5 bộ test.

In [ ]:
# 1) Lấy code đã sửa bug từ GitHub
!rm -rf Deepfake-Detect && git clone --depth 1 https://github.com/KimThanhTran/Deepfake-Detect.git
%cd Deepfake-Detect
!pip -q install scikit-learn tqdm 2>/dev/null
import torch; print('CUDA:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')

In [ ]:
# 2) Tìm dataset benchmark trong /kaggle/input
import glob, os
hits = sum((glob.glob(f'/kaggle/input/{d}/ForenSynths/ForenSynths/train') for d in ('*', '*/*', '*/*/*')), [])
assert hits, 'Không tìm thấy ForenSynths/ForenSynths/train trong /kaggle/input — hãy Add Input dataset benchmark'
DATA = os.path.dirname(os.path.dirname(os.path.dirname(hits[0])))
print('DATA =', DATA)
!ls {DATA}

In [ ]:
# 3) Train Hybrid: spatial NPR dong bang, chi train frequency branch + fusion
!python train_frequency.py \
    --spatial_model_path weights/NPR.pth \
    --dataroot {DATA}/ForenSynths/ForenSynths \
    --classes car cat chair horse \
    --batch_size 64 --epochs 10 --lr 0.001 --num_workers 4

In [ ]:
# 4) Lay checkpoint tot nhat
import glob, shutil
best = sorted(glob.glob('checkpoints/hybrid_frequency_*/model_best.pth'))[-1]
shutil.copy(best, '/kaggle/working/hybrid_best.pth')
hist = sorted(glob.glob('checkpoints/hybrid_frequency_*/training_history.csv'))[-1]
shutil.copy(hist, '/kaggle/working/training_history.csv')
print('best =', best)

In [ ]:
# 5) Danh gia Hybrid tren ca 5 bo test (metrics + confusion matrices)
import os
os.makedirs('/kaggle/working/results', exist_ok=True)
sets = {
    'ForenSynths-test':    f'{DATA}/ForenSynths/ForenSynths/test',
    'GANGen-Detection':    f'{DATA}/GANGen-Detection/GANGen-Detection',
    'UniversalFakeDetect': f'{DATA}/UniversalFakeDetect/UniversalFakeDetect',
    'DiffusionForensics':  f'{DATA}/DiffusionForensics/DiffusionForensics',
    'Diffusion1kStep':     f'{DATA}/Diffusion1kStep/Diffusion1kStep',
}
for name, root in sets.items():
    if not os.path.isdir(root):
        print('[skip]', name); continue
    !python tools/eval_report.py --arch hybrid \
        --spatial_model_path weights/NPR.pth \
        --model_path /kaggle/working/hybrid_best.pth \
        --dataroot {root} \
        --out_dir /kaggle/working/results/hybrid_{name} \
        --label "HSF Hybrid - {name}" --batch_size 64 --num_workers 4

In [ ]:
# 6) Tong hop
import pandas as pd, glob
for f in sorted(glob.glob('/kaggle/working/results/*/metrics.csv')):
    print('\n===', f.split('/')[-2])
    print(pd.read_csv(f).to_string(index=False))